# CASCADE — Region Comparison
Compare batch-fitting results across user-selected spatial regions.

**Workflow:**
1. Set `H5_FILE` in the **Configuration** cell and run all cells through *Region Selection*
2. Draw rectangular regions on the image, click **Add Region** after each, then **Done**
3. Run **Peak Extraction** → **Hungarian Matching** → **Class Assignment**
4. Run any visualisation cell in any order

**Visualisations available:**
| Cell | Plot | What it shows |
|---|---|---|
| VIZ 1 | Side-by-side lollipop | Per-peak amplitude, one panel per region |
| VIZ 2 | Overlaid lollipop | All regions on one axis, x-jittered |
| VIZ 3 | Differential lollipop | Δ amplitude between two regions |
| VIZ 4 | Violin by class | Within-region amplitude distribution per class |
| VIZ 5 | Box plots (matched) | Pixel-level amplitude spread for each matched peak |
| VIZ 6 | Spatial heatmaps | Where in the region a class/peak is strongest |
| VIZ 7 | Radar / spider chart | Class-level profile per region |
| VIZ 8 | Grouped bar + error bars | Class-level mean ± std per region |
| VIZ 9 | Prevalence lollipop | Head size = fraction of pixels that have the peak |

In [ ]:
import numpy as np
import h5py
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('widget')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.widgets import RectangleSelector, Button

from scipy.optimize import linear_sum_assignment
from scipy.cluster.hierarchy import fclusterdata

from IPython.display import display, clear_output

print('Imports OK')

In [ ]:
# ── File path ─────────────────────────────────────────────────────────────────
H5_FILE = 'fitted_output.h5'    # ← set to your .h5 (or .npz) path

# ── Detection thresholds ─────────────────────────────────────────────────────
AMP_THRESHOLD          = 0.01   # peaks below this amplitude are ignored
CENTER_MATCH_THRESHOLD = 15.0   # cm⁻¹ — peaks closer than this are the same peak

# ── Peak classes ─────────────────────────────────────────────────────────────
# Edit wavenumber ranges (cm⁻¹) to match your spectral features.
# Any number of classes is supported.
PEAK_CLASSES = {
    'Class A (800–1000)':  ( 800, 1000),
    'Class B (1000–1200)': (1000, 1200),
    'Class C (1200–1400)': (1200, 1400),
    'Class D (1400–1650)': (1400, 1650),
}

REGION_COLORS = [
    'tab:blue', 'tab:orange', 'tab:green', 'tab:red',
    'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray',
]

In [ ]:
def _load_h5(path):
    with h5py.File(path, 'r') as f:
        g   = f['preprocessed_images']
        pp  = g['peak_params'][:]
        xa  = g['x_axis'][:]
        mdl = g['model'][:] if 'model' in g else None
        raw = g['raw'][:]   if 'raw'   in g else None
    return pp, xa, mdl, raw

def _load_npz(path):
    d = np.load(path, allow_pickle=True)
    return d['peak_params'], d['x_axis'], d.get('model'), d.get('raw')

ext = H5_FILE.rsplit('.', 1)[-1].lower()
if ext in ('h5', 'hdf5'):
    peak_params, x_axis, model, raw = _load_h5(H5_FILE)
else:
    peak_params, x_axis, model, raw = _load_npz(H5_FILE)

H, W, P   = peak_params.shape
max_peaks = P // 4
peaks_4d  = peak_params.reshape(H, W, max_peaks, 4)
# peaks_4d[y, x, k, 0] = amplitude  (0–1, normalised)
# peaks_4d[y, x, k, 1] = center     (cm⁻¹)
# peaks_4d[y, x, k, 2] = sigma      (cm⁻¹, Gaussian half-width)
# peaks_4d[y, x, k, 3] = gamma      (cm⁻¹, Lorentzian half-width)

if model is not None:
    display_img = model.mean(axis=2)
elif raw is not None:
    display_img = raw.astype(float).mean(axis=2)
else:
    display_img = peaks_4d[:, :, :, 0].max(axis=2)

wn_lo, wn_hi = float(x_axis.min()), float(x_axis.max())
print(f'Loaded {H}×{W} pixels  |  max_peaks={max_peaks}  |  WN {wn_lo:.0f}–{wn_hi:.0f} cm⁻¹')

## Region Selection
1. Click and drag on the image to draw a rectangle
2. Click **Add Region** to confirm it
3. Repeat for each region you want to compare
4. Click **Done** when finished — then run the next cell

In [ ]:
regions       = []       # populated by the UI buttons below
_pending      = [None]   # rectangle drawn but not yet confirmed
_live_patch   = [None]   # current dashed rectangle artist

fig_sel, ax_img = plt.subplots(figsize=(8, 6))
im = ax_img.imshow(display_img, cmap='inferno', aspect='auto')
ax_img.set_title('Draw rectangles — Add Region after each, then Done')
plt.colorbar(im, ax=ax_img, fraction=0.03, label='Intensity')

def _on_rect_select(eclick, erelease):
    r0 = max(0,   int(min(eclick.ydata, erelease.ydata) + 0.5))
    r1 = min(H-1, int(max(eclick.ydata, erelease.ydata) + 0.5))
    c0 = max(0,   int(min(eclick.xdata, erelease.xdata) + 0.5))
    c1 = min(W-1, int(max(eclick.xdata, erelease.xdata) + 0.5))
    color = REGION_COLORS[len(regions) % len(REGION_COLORS)]
    if _live_patch[0] is not None:
        try:
            _live_patch[0].remove()
        except Exception:
            pass
    patch = mpatches.Rectangle(
        (c0, r0), c1 - c0, r1 - r0,
        linewidth=2, edgecolor=color, facecolor='none', linestyle='--')
    ax_img.add_patch(patch)
    _live_patch[0] = patch
    _pending[0] = dict(rows=(r0, r1), cols=(c0, c1), color=color)
    fig_sel.canvas.draw_idle()

_rs = RectangleSelector(
    ax_img, _on_rect_select, useblit=True,
    button=[1], minspanx=2, minspany=2,
    spancoords='pixels', interactive=False)

def _btn_add(_):
    if _pending[0] is None:
        print('Draw a rectangle first, then click Add Region.')
        return
    name = f'Region {len(regions) + 1}'
    reg  = dict(name=name, **_pending[0])
    regions.append(reg)
    _pending[0] = None
    if _live_patch[0] is not None:
        _live_patch[0].set_linestyle('-')
        _live_patch[0] = None
    fig_sel.canvas.draw_idle()
    rows_s = str(reg['rows'])
    cols_s = str(reg['cols'])
    print(f'Added {name}  rows={rows_s}  cols={cols_s}')

def _btn_clear(_):
    regions.clear()
    for p in list(ax_img.patches):
        p.remove()
    _pending[0] = None
    _live_patch[0] = None
    fig_sel.canvas.draw_idle()
    print('Cleared all regions.')

def _btn_done(_):
    if _pending[0] is not None:
        _btn_add(None)
    print(f'{len(regions)} region(s) confirmed — run the next cell.')

ax_add   = plt.axes([0.10, 0.01, 0.24, 0.055])
ax_clear = plt.axes([0.38, 0.01, 0.24, 0.055])
ax_done  = plt.axes([0.66, 0.01, 0.24, 0.055])
btn_add   = Button(ax_add,   'Add Region')
btn_clear = Button(ax_clear, 'Clear All')
btn_done  = Button(ax_done,  'Done')
btn_add.on_clicked(_btn_add)
btn_clear.on_clicked(_btn_clear)
btn_done.on_clicked(_btn_done)
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)
plt.show()

## Peak Extraction and Matching
Run cells in order: **Extract → Match → Classify**

In [ ]:
def _get_region_peaks(rows, cols):
    r0, r1 = rows
    c0, c1 = cols
    sub  = peaks_4d[r0:r1+1, c0:c1+1, :, :]   # (rH, rW, P, 4)
    amps = sub[..., 0]
    ctrs = sub[..., 1]
    sigs = sub[..., 2]
    gams = sub[..., 3]
    mask = amps > AMP_THRESHOLD
    n_px = (r1 - r0 + 1) * (c1 - c0 + 1)
    return dict(
        amp=amps[mask], center=ctrs[mask],
        sigma=sigs[mask], gamma=gams[mask],
        pixel_amps=amps, pixel_ctrs=ctrs, pixel_mask=mask,
        n_pixels=n_px, sub_shape=sub.shape[:2],
    )

def _cluster_peaks(peaks_dict, threshold=CENTER_MATCH_THRESHOLD):
    centers = peaks_dict['center']
    if len(centers) == 0:
        return []
    if len(centers) == 1:
        labels = np.array([1])
    else:
        labels = fclusterdata(
            centers.reshape(-1, 1), t=threshold,
            criterion='distance', method='average')
    amps   = peaks_dict['amp']
    sigmas = peaks_dict['sigma']
    gammas = peaks_dict['gamma']
    n_px   = peaks_dict['n_pixels']
    result = []
    for lbl in np.unique(labels):
        idx = labels == lbl
        fwhm_g = 2.35482 * sigmas[idx].mean()
        fwhm_l = 2.0     * gammas[idx].mean()
        result.append(dict(
            mean_center = float(centers[idx].mean()),
            std_center  = float(centers[idx].std()),
            mean_amp    = float(amps[idx].mean()),
            std_amp     = float(amps[idx].std()),
            mean_sigma  = float(sigmas[idx].mean()),
            mean_gamma  = float(gammas[idx].mean()),
            mean_fwhm   = float(fwhm_g + fwhm_l),
            prevalence  = float(idx.sum()) / n_px,
            n_obs       = int(idx.sum()),
            cls         = None,
        ))
    result.sort(key=lambda d: d['mean_center'])
    return result

assert len(regions) >= 2, 'Select at least 2 regions before running this cell.'

for reg in regions:
    pd_ = _get_region_peaks(reg['rows'], reg['cols'])
    reg['peaks_dict'] = pd_
    reg['consensus']  = _cluster_peaks(pd_)
    name = reg['name']
    n_c  = len(reg['consensus'])
    n_o  = pd_['amp'].size
    n_px = pd_['n_pixels']
    print(f'{name}: {n_c} consensus peaks from {n_o} observations ({n_px} pixels)')

In [ ]:
def match_peaks(reg_a, reg_b, threshold=CENTER_MATCH_THRESHOLD):
    """Hungarian matching of consensus peaks between two regions."""
    ca_list = reg_a['consensus']
    cb_list = reg_b['consensus']
    if not ca_list or not cb_list:
        return []
    ca   = np.array([p['mean_center'] for p in ca_list])
    cb   = np.array([p['mean_center'] for p in cb_list])
    cost = np.abs(ca[:, None] - cb[None, :])
    pen  = np.where(cost <= threshold, cost, 1e9)
    ri, ci = linear_sum_assignment(pen)
    matches = []
    for r, c in zip(ri, ci):
        if cost[r, c] <= threshold:
            matches.append(dict(
                idx_a=int(r), idx_b=int(c),
                peak_a=ca_list[r], peak_b=cb_list[c],
                center_a=float(ca[r]), center_b=float(cb[c]),
                delta_center=float(cb[c] - ca[r]),
                cost=float(cost[r, c]),
            ))
    return matches

ref = regions[0]
for reg in regions[1:]:
    reg['matches_vs_ref'] = match_peaks(ref, reg)
    n_ref   = len(ref['consensus'])
    n_other = len(reg['consensus'])
    n_m     = len(reg['matches_vs_ref'])
    ref_n   = ref['name']
    other_n = reg['name']
    print(f'{ref_n} ({n_ref} peaks)  ↔  {other_n} ({n_other} peaks):  {n_m} matched')

In [ ]:
def _assign_classes(consensus, class_def=PEAK_CLASSES):
    for pk in consensus:
        pk['cls'] = None
        for cls, (lo, hi) in class_def.items():
            if lo <= pk['mean_center'] < hi:
                pk['cls'] = cls
                break

def _class_summary(reg, class_def=PEAK_CLASSES):
    r0, r1 = reg['rows']
    c0, c1 = reg['cols']
    sa  = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
    sc  = peaks_4d[r0:r1+1, c0:c1+1, :, 1]
    n_px = reg['peaks_dict']['n_pixels']
    summary = {}
    for cls, (lo, hi) in class_def.items():
        mask    = (sa > AMP_THRESHOLD) & (sc >= lo) & (sc < hi)
        vals    = sa[mask]
        members = [p for p in reg['consensus'] if p.get('cls') == cls]
        summary[cls] = dict(
            n_peaks    = len(members),
            total_amp  = float(vals.sum() / n_px) if len(vals) > 0 else 0.0,
            mean_amp   = float(vals.mean())        if len(vals) > 0 else 0.0,
            std_amp    = float(vals.std())         if len(vals) > 0 else 0.0,
            max_amp    = float(vals.max())         if len(vals) > 0 else 0.0,
            prevalence = float(mask.any(axis=2).mean()),
        )
    return summary

for reg in regions:
    _assign_classes(reg['consensus'])
    reg['class_summary'] = _class_summary(reg)

print('Class assignment complete.  Summary (mean_amp | prevalence):')
col_w = 22
hdr = 'Class'.ljust(36) + ''.join(r['name'].rjust(col_w) for r in regions)
print(hdr)
print('-' * len(hdr))
for cls in PEAK_CLASSES:
    row = cls.ljust(36)
    for reg in regions:
        v   = reg['class_summary'][cls]
        row += f'{v["mean_amp"]:.4f}/{v["prevalence"]:.1%}'.rjust(col_w)
    print(row)

## Visualisations
Each cell below is independent — run them in any order.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 1 — Side-by-side lollipop
#  One panel per region; head colour = peak class; error bar = std_amp
# ═══════════════════════════════════════════════════════════════════

def _cls_palette():
    keys   = list(PEAK_CLASSES.keys())
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(keys), 1)))
    return {k: colors[i] for i, k in enumerate(keys)}

def plot_lollipop(regions, metric='mean_amp', figsize=None):
    if figsize is None:
        figsize = (5 * len(regions), 5)
    palette = _cls_palette()
    all_vals = [p[metric] for r in regions for p in r['consensus']]
    y_max    = max(all_vals) if all_vals else 1.0

    fig, axes = plt.subplots(1, len(regions), figsize=figsize, sharey=True)
    if len(regions) == 1:
        axes = [axes]

    for ax, reg in zip(axes, regions):
        for pk in reg['consensus']:
            x     = pk['mean_center']
            y     = pk[metric]
            color = palette.get(pk.get('cls'), 'lightgray')
            ax.vlines(x, 0, y, color=reg['color'], lw=1.5, alpha=0.65)
            ax.scatter(x, y, s=55, color=color, zorder=5,
                       edgecolors='k', linewidths=0.4)
            if pk['std_amp'] > 0:
                ax.errorbar(x, y, yerr=pk['std_amp'], fmt='none',
                            ecolor='gray', elinewidth=0.8, capsize=2)
        ax.set_xlim(wn_lo - 15, wn_hi + 15)
        ax.set_ylim(0, y_max * 1.15)
        ax.set_title(reg['name'], color=reg['color'], fontweight='bold')
        ax.set_xlabel('Wavenumber (cm⁻¹)')
        ax.spines[['top', 'right']].set_visible(False)

    axes[0].set_ylabel(metric.replace('_', ' ').title())
    handles = [mpatches.Patch(color=palette[k], label=k) for k in PEAK_CLASSES]
    fig.legend(handles=handles, loc='upper right', fontsize=8,
               title='Peak Class', framealpha=0.85)
    metric_lbl = metric.replace('_', ' ').title()
    fig.suptitle(f'Lollipop — {metric_lbl}', fontsize=13)
    plt.tight_layout()
    plt.show()

plot_lollipop(regions, metric='mean_amp')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 2 — Overlaid lollipop
#  All regions on one axis; peaks are x-jittered for readability
# ═══════════════════════════════════════════════════════════════════

def plot_lollipop_overlay(regions, metric='mean_amp', figsize=(12, 5)):
    palette = _cls_palette()
    n      = len(regions)
    jitter = np.linspace(-2.5, 2.5, n) if n > 1 else [0.0]

    fig, ax = plt.subplots(figsize=figsize)
    for reg, jit in zip(regions, jitter):
        for pk in reg['consensus']:
            x     = pk['mean_center'] + jit
            y     = pk[metric]
            color = palette.get(pk.get('cls'), 'lightgray')
            ax.vlines(x, 0, y, color=reg['color'], lw=1.5, alpha=0.6)
            ax.scatter(x, y, s=48, color=color, zorder=5,
                       edgecolors=reg['color'], linewidths=0.8)

    reg_handles = [mpatches.Patch(color=r['color'], label=r['name']) for r in regions]
    cls_handles = [mpatches.Patch(color=palette[k], label=k) for k in PEAK_CLASSES]
    leg1 = ax.legend(handles=reg_handles, loc='upper left',  fontsize=8, title='Region')
    ax.add_artist(leg1)
    ax.legend(handles=cls_handles, loc='upper right', fontsize=8, title='Class')
    ax.set_xlabel('Wavenumber (cm⁻¹)')
    ax.set_ylabel(metric.replace('_', ' ').title())
    metric_lbl = metric.replace('_', ' ').title()
    ax.set_title(f'Overlaid Lollipop — {metric_lbl}')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_lollipop_overlay(regions, metric='mean_amp')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 3 — Differential lollipop  (region B − region A)
#  Positive = peak stronger in B; negative = stronger in A
#  Change REG_A_IDX / REG_B_IDX to compare any pair
# ═══════════════════════════════════════════════════════════════════

REG_A_IDX = 0   # index into `regions` list
REG_B_IDX = 1

def plot_diff_lollipop(reg_a, reg_b, figsize=(12, 5)):
    matches = match_peaks(reg_a, reg_b)
    if not matches:
        print('No matched peaks — try increasing CENTER_MATCH_THRESHOLD.')
        return
    fig, ax = plt.subplots(figsize=figsize)
    palette = _cls_palette()
    for m in matches:
        x     = (m['center_a'] + m['center_b']) / 2
        dy    = m['peak_b']['mean_amp'] - m['peak_a']['mean_amp']
        stem_c = reg_b['color'] if dy >= 0 else reg_a['color']
        cls   = m['peak_a'].get('cls') or m['peak_b'].get('cls')
        head_c = palette.get(cls, 'lightgray')
        ax.vlines(x, 0, dy, color=stem_c, lw=2, alpha=0.8)
        ax.scatter(x, dy, s=60, color=head_c, zorder=5,
                   edgecolors=stem_c, linewidths=0.8)

    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_xlabel('Wavenumber (cm⁻¹)')
    name_a = reg_a['name']
    name_b = reg_b['name']
    ax.set_ylabel(f'Δ Amplitude  ({name_b} − {name_a})')
    ax.set_title(f'Differential Lollipop: {name_a}  vs  {name_b}')
    ax.text(0.01, 0.97, f'▲ stronger in {name_b}',
            transform=ax.transAxes, color=reg_b['color'], fontsize=9, va='top')
    ax.text(0.01, 0.03, f'▼ stronger in {name_a}',
            transform=ax.transAxes, color=reg_a['color'], fontsize=9, va='bottom')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_diff_lollipop(regions[REG_A_IDX], regions[REG_B_IDX])

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 4 — Violin plots by peak class
#  Shows the pixel-level amplitude distribution within each region
# ═══════════════════════════════════════════════════════════════════

def plot_violin_by_class(regions, class_def=PEAK_CLASSES, figsize=None):
    cls_list = list(class_def.keys())
    if figsize is None:
        figsize = (max(4 * len(cls_list), 8), 5)
    fig, axes = plt.subplots(1, len(cls_list), figsize=figsize, sharey=False)
    if len(cls_list) == 1:
        axes = [axes]

    for ax, cls in zip(axes, cls_list):
        lo, hi = class_def[cls]
        data, labels = [], []
        for reg in regions:
            r0, r1 = reg['rows']
            c0, c1 = reg['cols']
            sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
            sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]
            mask = (sa > AMP_THRESHOLD) & (sc >= lo) & (sc < hi)
            vals = sa[mask]
            data.append(vals if len(vals) > 1 else np.array([0.0, 0.0]))
            labels.append(reg['name'])

        parts = ax.violinplot(data, showmedians=True, showextrema=True)
        for pc, reg in zip(parts['bodies'], regions):
            pc.set_facecolor(reg['color'])
            pc.set_alpha(0.6)
        for key in ('cmedians', 'cmaxes', 'cmins', 'cbars'):
            if key in parts:
                parts[key].set_color('black')
                parts[key].set_linewidth(0.8)
        ax.set_xticks(range(1, len(regions) + 1))
        ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=8)
        ax.set_title(cls, fontsize=9)
        ax.spines[['top', 'right']].set_visible(False)

    axes[0].set_ylabel('Peak Amplitude')
    fig.suptitle('Amplitude Distribution by Peak Class and Region', fontsize=13)
    plt.tight_layout()
    plt.show()

plot_violin_by_class(regions)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 5 — Box plots for matched peaks
#  Each box = pixel-level amplitude distribution for one matched peak
#  X-axis labels show the reference region's peak center
# ═══════════════════════════════════════════════════════════════════

def plot_boxplot_matched(regions, figsize=(13, 5)):
    if len(regions) < 2:
        print('Need at least 2 regions.')
        return
    ref     = regions[0]
    ref_pks = ref['consensus']
    if not ref_pks:
        print('Reference region has no consensus peaks.')
        return

    n_pk  = len(ref_pks)
    n_reg = len(regions)
    width = 0.8 / n_reg
    offs  = np.linspace(-0.4 + width / 2, 0.4 - width / 2, n_reg)
    x_pos = np.arange(n_pk)
    fig, ax = plt.subplots(figsize=figsize)

    def _pixel_amps(reg, center_wn):
        r0, r1 = reg['rows']
        c0, c1 = reg['cols']
        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]
        m  = (sa > AMP_THRESHOLD) & (np.abs(sc - center_wn) < CENTER_MATCH_THRESHOLD)
        return sa[m] if m.any() else np.array([0.0])

    for reg, off in zip(regions, offs):
        per_peak = []
        if reg is ref:
            for pk in ref_pks:
                per_peak.append(_pixel_amps(reg, pk['mean_center']))
        else:
            mm = {m['idx_a']: m for m in reg.get('matches_vs_ref', [])}
            for i, pk in enumerate(ref_pks):
                ctr = mm[i]['center_b'] if i in mm else None
                per_peak.append(_pixel_amps(reg, ctr) if ctr is not None else np.array([0.0]))
        bp = ax.boxplot(
            per_peak, positions=x_pos + off, widths=width * 0.85,
            patch_artist=True, manage_ticks=False,
            medianprops=dict(color='black', lw=1.5),
            whiskerprops=dict(color=reg['color']),
            capprops=dict(color=reg['color']),
            flierprops=dict(marker='.', markerfacecolor=reg['color'],
                            markersize=3, alpha=0.4))
        for patch in bp['boxes']:
            patch.set_facecolor(reg['color'])
            patch.set_alpha(0.55)

    ax.set_xticks(x_pos)
    tick_labels = [f'{p["mean_center"]:.0f} cm\u207b\u00b9' for p in ref_pks]
    ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Peak Amplitude')
    ax.set_title('Matched Peak Amplitude Distribution per Region')
    handles = [mpatches.Patch(color=r['color'], label=r['name']) for r in regions]
    ax.legend(handles=handles, fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_boxplot_matched(regions)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 6 — Spatial heatmaps
#  Shows where within each region a given class or peak is strongest
#  Colourscale is shared across regions for direct comparison
# ═══════════════════════════════════════════════════════════════════

def plot_spatial_heatmaps(
        regions,
        target='class',          # 'class', 'peak', or 'max'
        class_name=None,         # string key from PEAK_CLASSES
        peak_center=None,        # wavenumber (cm⁻¹) for target='peak'
        cmap='plasma',
        figsize=None):
    n = len(regions)
    if figsize is None:
        figsize = (4 * n, 4)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]

    maps, vmin, vmax = [], np.inf, -np.inf
    for reg in regions:
        r0, r1 = reg['rows']
        c0, c1 = reg['cols']
        sa = peaks_4d[r0:r1+1, c0:c1+1, :, 0]
        sc = peaks_4d[r0:r1+1, c0:c1+1, :, 1]
        if target == 'class' and class_name in PEAK_CLASSES:
            lo, hi = PEAK_CLASSES[class_name]
            mask = (sa > AMP_THRESHOLD) & (sc >= lo) & (sc < hi)
            amp_map = (sa * mask).sum(axis=2)
        elif target == 'peak' and peak_center is not None:
            mask = (sa > AMP_THRESHOLD) & \
                   (np.abs(sc - peak_center) < CENTER_MATCH_THRESHOLD)
            amp_map = (sa * mask).max(axis=2)
        else:
            amp_map = sa.max(axis=2)
        maps.append(amp_map)
        vmin = min(vmin, amp_map.min())
        vmax = max(vmax, amp_map.max())

    for ax, reg, amp_map in zip(axes, regions, maps):
        im = ax.imshow(amp_map, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
        ax.set_title(reg['name'], color=reg['color'], fontweight='bold')
        ax.set_xlabel('Col')
        ax.set_ylabel('Row')
        plt.colorbar(im, ax=ax, shrink=0.8, label='Amplitude')

    if class_name:
        title = f'Spatial Heatmap — {class_name}'
    elif peak_center is not None:
        title = f'Spatial Heatmap — {peak_center:.0f} cm\u207b\u00b9'
    else:
        title = 'Spatial Heatmap — Max Peak Amplitude'
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

# Plot each class separately
for cls_name in PEAK_CLASSES:
    plot_spatial_heatmaps(regions, target='class', class_name=cls_name)

# Uncomment to plot a specific peak by wavenumber:
# plot_spatial_heatmaps(regions, target='peak', peak_center=1300.0)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 7 — Radar / spider chart
#  One axis per class; one polygon per region
#  Good for quickly seeing which region is dominant in which class
# ═══════════════════════════════════════════════════════════════════

def plot_radar(regions, metric='mean_amp', figsize=(7, 7)):
    cls_list = list(PEAK_CLASSES.keys())
    N = len(cls_list)
    if N < 3:
        print('Radar chart needs at least 3 classes — add more to PEAK_CLASSES.')
        return
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    closed = angles + angles[:1]

    all_vals = [reg['class_summary'][cls][metric]
                for reg in regions for cls in cls_list]
    scale = max(all_vals) if max(all_vals) > 0 else 1.0

    fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(polar=True))
    for reg in regions:
        vals = [reg['class_summary'][cls][metric] / scale for cls in cls_list]
        vals_c = vals + vals[:1]
        ax.plot(closed, vals_c, color=reg['color'], lw=2, label=reg['name'])
        ax.fill(closed, vals_c, color=reg['color'], alpha=0.12)

    ax.set_xticks(angles)
    ax.set_xticklabels(cls_list, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=7)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
    metric_lbl = metric.replace('_', ' ').title()
    ax.set_title(f'Radar — {metric_lbl}\n(normalised to max across regions)',
                 fontsize=12, pad=20)
    plt.tight_layout()
    plt.show()

plot_radar(regions, metric='mean_amp')
plot_radar(regions, metric='prevalence')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 8 — Grouped bar chart with error bars  (class level)
#  Error bars = std of per-pixel amplitude within the class
# ═══════════════════════════════════════════════════════════════════

def plot_bar_classes(regions, metric='mean_amp', figsize=(11, 5)):
    cls_list = list(PEAK_CLASSES.keys())
    n_cls = len(cls_list)
    n_reg = len(regions)
    x     = np.arange(n_cls)
    width = 0.8 / n_reg
    offs  = np.linspace(-0.4 + width / 2, 0.4 - width / 2, n_reg)

    fig, ax = plt.subplots(figsize=figsize)
    for reg, off in zip(regions, offs):
        vals = [reg['class_summary'][cls][metric] for cls in cls_list]
        errs = [reg['class_summary'][cls]['std_amp']  for cls in cls_list]
        ax.bar(x + off, vals, width=width, color=reg['color'], alpha=0.75,
               label=reg['name'], yerr=errs, capsize=3,
               error_kw=dict(elinewidth=1, ecolor='dimgray'))

    ax.set_xticks(x)
    ax.set_xticklabels(cls_list, rotation=15, ha='right', fontsize=9)
    metric_lbl = metric.replace('_', ' ').title()
    ax.set_ylabel(metric_lbl)
    ax.set_title(f'Class-Level {metric_lbl} per Region')
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_bar_classes(regions, metric='mean_amp')
plot_bar_classes(regions, metric='prevalence')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  VIZ 9 — Prevalence lollipop
#  Stem height = mean amplitude; head size = fraction of pixels
#  that contain that peak; head colour = peak class
# ═══════════════════════════════════════════════════════════════════

def plot_prevalence_lollipop(regions, figsize=(12, 5)):
    palette = _cls_palette()
    n      = len(regions)
    jitter = np.linspace(-3.0, 3.0, n) if n > 1 else [0.0]

    fig, ax = plt.subplots(figsize=figsize)
    for reg, jit in zip(regions, jitter):
        for pk in reg['consensus']:
            x      = pk['mean_center'] + jit
            y      = pk['mean_amp']
            sz     = 20 + 200 * pk['prevalence']   # size ∝ prevalence
            color  = palette.get(pk.get('cls'), 'lightgray')
            ax.vlines(x, 0, y, color=reg['color'], lw=1.2, alpha=0.55)
            ax.scatter(x, y, s=sz, color=color, zorder=5,
                       edgecolors=reg['color'], linewidths=0.8, alpha=0.85)

    reg_handles = [mpatches.Patch(color=r['color'], label=r['name']) for r in regions]
    leg1 = ax.legend(handles=reg_handles, loc='upper left',
                     fontsize=8, title='Region')
    ax.add_artist(leg1)
    cls_handles = [mpatches.Patch(color=palette[k], label=k) for k in PEAK_CLASSES]
    ax.legend(handles=cls_handles, loc='upper right', fontsize=8, title='Class')
    ax.set_xlabel('Wavenumber (cm⁻¹)')
    ax.set_ylabel('Mean Amplitude')
    ax.set_title('Prevalence Lollipop  —  head size ∝ fraction of pixels with peak')
    ax.spines[['top', 'right']].set_visible(False)
    ax.text(0.99, 0.01, '● common   ○ rare',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, color='gray')
    plt.tight_layout()
    plt.show()

plot_prevalence_lollipop(regions)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Text summary table
# ═══════════════════════════════════════════════════════════════════

def print_summary_table(regions, col_w=15):
    ref     = regions[0]
    ref_pks = ref['consensus']
    sep_w   = 36 + col_w * len(regions)

    print('=' * sep_w)
    print('MATCHED PEAK SUMMARY  (mean_amp, relative to reference region)')
    print('=' * sep_w)
    hdr = 'Center (cm⁻¹)  Class'.ljust(36)
    hdr += ''.join(r['name'].rjust(col_w) for r in regions)
    print(hdr)
    print('-' * sep_w)

    match_maps = [{}] + [{m['idx_a']: m for m in r.get('matches_vs_ref', [])}
                         for r in regions[1:]]
    for i, pk in enumerate(ref_pks):
        cls_str = str(pk.get('cls') or '-')
        row = f'{pk["mean_center"]:>8.1f}  {cls_str:<26}'
        row += f'{pk["mean_amp"]:>{col_w}.4f}'
        for mm in match_maps[1:]:
            if i in mm:
                amp_v = mm[i]['peak_b']['mean_amp']
                row  += f'{amp_v:>{col_w}.4f}'
            else:
                row  += '(absent)'.rjust(col_w)
        print(row)

    print()
    print('=' * sep_w)
    print('CLASS SUMMARY  (mean_amp | prevalence)')
    print('=' * sep_w)
    hdr2 = 'Class'.ljust(36)
    hdr2 += ''.join(r['name'].rjust(col_w) for r in regions)
    print(hdr2)
    print('-' * sep_w)
    for cls in PEAK_CLASSES:
        row2 = cls.ljust(36)
        for reg in regions:
            v     = reg['class_summary'][cls]
            entry = f'{v["mean_amp"]:.3f}/{v["prevalence"]:.1%}'
            row2 += entry.rjust(col_w)
        print(row2)
    print()

print_summary_table(regions)